# Crypto-AML-Analysis — USDT Informal-Exchanger & CTF Lead Detection

**Single self-contained notebook.** Detects, as *research leads*, independent/unregulated USDT exchangers
("funnel accounts") and tests counter-terror-financing (CTF) proximity to OFAC-sanctioned addresses — on
**both** Ethereum (ERC-20) and **Tron (TRC-20)**, via Google BigQuery public datasets.

This notebook inlines every layer (data → graph → features → anomaly ML → enrichment → CTF → viz), so it runs
on its own with no local `src/` modules. It folds in two refinements proven against live data:
1. **Dust floor** — require a minimum *median* per-transfer value, to drop airdrop/dust-spam false positives.
2. **Tron/TRC-20 path** — Tron carries most real USDT volume; its transfers live in `decoded_events`.

---
### What an executed run already found (Ethereum, 90 days; Tron, sampled)
- **971** EOA funnel candidates on Ethereum (top wallet: 45,413 distinct senders, ~1 outgoing).
- **CTF dead-end on listed addresses:** across *all* USDT transfers, the 182 OFAC ETH/USDT addresses had **1**
  transfer in 90 days (0 touching candidates); on Tron, **1** of **71M** transfers in 30 days. **Tether
  freezes OFAC-listed addresses**, so they go dormant — direct proximity to *already-listed* addresses yields
  almost nothing. Value is in funnel detection + behavioural/upstream tracing.
- **0/971** leads matched the public Etherscan label set → unlabeled (independent) operators.
- **Tron** has far more funnel activity (2,646 candidates in just 7 days, dust-floored).

> ⚠️ **Leads, not proof.** High fan-in / low fan-out also fits legitimate processors, OTC desks and exchange
> deposit wallets. Corroborate independently. Public on-chain data only. Use authoritative OFAC seeds only.

## 1. Setup & authentication

```bash
pip install -r requirements.txt
```

Configure your credentials in a local `.env` file (copied from `.env.example`):
```
BQ_PROJECT=your-gcp-project-id
BQ_ACCESS_TOKEN=ya29...      # optional, from `gcloud auth print-access-token`
```

Or authenticate persistently with `gcloud auth application-default login` and leave `BQ_ACCESS_TOKEN` blank.

> 🔒 **Never commit `.env`** — it is git-ignored.

In [ ]:
import os, json, time, urllib.request
import numpy as np, pandas as pd
from google.cloud import bigquery

# ============================================================================
#  AUTHENTICATION — set these from your environment / .env file
#
#  Get a token on your machine with:   gcloud auth print-access-token
#  Or run once for persistent auth :   gcloud auth application-default login
#
#  Required:  BQ_PROJECT       (your billable GCP project ID)
#  Optional:  BQ_ACCESS_TOKEN  (~1h short-lived token; blank uses ADC)
# ============================================================================
BQ_ACCESS_TOKEN = os.environ.get("BQ_ACCESS_TOKEN", "").strip()
BQ_PROJECT      = os.environ.get("BQ_PROJECT", "").strip()

# Fallback: try loading from .env file in the same folder as this notebook
if not BQ_PROJECT:
    env_path = os.path.join(os.path.dirname(os.path.abspath("")), ".env")
    if not os.path.exists(env_path):
        env_path = ".env"
    if os.path.exists(env_path):
        for line in open(env_path, encoding="utf-8"):
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, _, v = line.partition("=")
            v = v.strip().strip('"').strip("'")
            if k.strip() == "BQ_PROJECT" and not BQ_PROJECT:
                BQ_PROJECT = v
            if k.strip() == "BQ_ACCESS_TOKEN" and not BQ_ACCESS_TOKEN:
                BQ_ACCESS_TOKEN = v

if not BQ_PROJECT:
    raise RuntimeError(
        "BQ_PROJECT is not set.\n"
        "Create a .env file next to this notebook with:\n"
        "    BQ_PROJECT=your-gcp-project-id\n"
        "    BQ_ACCESS_TOKEN=ya29...     (optional, from `gcloud auth print-access-token`)\n"
        "See .env.example for the template."
    )

if BQ_ACCESS_TOKEN:
    from google.oauth2.credentials import Credentials
    client = bigquery.Client(project=BQ_PROJECT, credentials=Credentials(token=BQ_ACCESS_TOKEN))
    print("Auth: BQ_ACCESS_TOKEN")
else:
    client = bigquery.Client(project=BQ_PROJECT)
    print("Auth: gcloud Application Default Credentials")
print("BigQuery client ready. Billing project:", client.project)

## Config — datasets, contracts, thresholds (incl. dust floor)

In [ ]:
# ---- BigQuery datasets ----
ETH_DATASET        = "bigquery-public-data.crypto_ethereum"
ETH_TOKEN_TRANSFERS = f"{ETH_DATASET}.token_transfers"
ETH_CONTRACTS      = f"{ETH_DATASET}.contracts"
TRON_DATASET       = "bigquery-public-data.goog_blockchain_tron_mainnet_us"
TRON_EVENTS        = f"{TRON_DATASET}.decoded_events"

# ---- USDT contracts ----
USDT_ERC20 = "0xdac17f958d2ee523a2206206994597c13d831ec7"             # Ethereum (lowercase)
USDT_TRC20_HEX = "0xa614f803b6fd780986a42c78ec9c7f77e6ded13c"         # Tron USDT in decoded_events (hex form)
USDT_TRC20_BASE58 = "TR7NHqjeKQxGTCi8q8ZY4pL8otSzgjLj6T"              # same contract, base58
USDT_DECIMALS = 6

# ---- Detection window & thresholds ----
LOOKBACK_DAYS         = int(os.environ.get("BQ_LOOKBACK_DAYS", 90))
RESULT_LIMIT          = 1000
MIN_DISTINCT_SENDERS  = 50       # high fan-in
MAX_OUT_TX            = 25       # low fan-out
MIN_IN_USDT           = 10_000   # meaningful inbound volume
MIN_MEDIAN_USDT       = 50       # 💡 DUST FLOOR: median per-transfer >= this (kills airdrop/dust spam)
MAX_GB_PER_QUERY      = float(os.environ.get("BQ_MAX_GB", 150))

# Known regulated venues to exclude (extend with your own labels).
KNOWN_EXCHANGE_ADDRESSES_ETH = {
    "0x28c6c06298d514db089934071355e5743bf21d60","0x21a31ee1afc51d94c2efccaa2092ad1028285549",
    "0xdfd5293d8e347dfe59e90efd55b2956a1343963d","0x267be1c1d684f78cb4f6a176c4911b741e4ffdc0",
    "0x6cc5f688a315f3dc28a7781717a9a798a59fda7b","0x876eabf441b2ee5b5b0554fd502a8e0600950cfa",
}
print(f"Window: {LOOKBACK_DAYS}d | dust floor median>=${MIN_MEDIAN_USDT} | byte ceiling {MAX_GB_PER_QUERY} GB")

## 2. Cost-safe query helpers

`token_transfers` (ETH) and `decoded_events` (Tron) are **partitioned by `block_timestamp`** — always keep the
`block_timestamp >= since` filter; it prunes partitions and is the biggest cost lever. Every query is dry-run
estimated and capped with `maximum_bytes_billed`.

In [ ]:
BYTES_PER_TB, PRICE_PER_TB_USD = 1024**4, 6.25

def estimate_cost(sql: str) -> float:
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed/1024**3
    print(f"Dry run: {gb:,.3f} GB (~${job.total_bytes_processed/BYTES_PER_TB*PRICE_PER_TB_USD:,.4f})")
    return gb

def run_query(sql: str, max_gb: float | None = None) -> pd.DataFrame:
    max_gb = MAX_GB_PER_QUERY if max_gb is None else max_gb
    gb = estimate_cost(sql)
    if gb > max_gb:
        raise RuntimeError(f"Would scan {gb:,.2f} GB > {max_gb} GB ceiling — tighten filters/window.")
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(max_gb*1024**3))
    return client.query(sql, job_config=cfg).to_dataframe(create_bqstorage_client=False)

## 3. OFAC sanctioned seeds (authoritative CTF anchors) + Tron base58→hex

Loaded from the community-maintained `0xB10C/ofac-sanctioned-digital-currency-addresses` list (tracks the
official OFAC SDN "Digital Currency Address" features). Tron seeds are base58 but the Tron dataset stores hex,
so we convert them.

In [ ]:
_OFAC = ("https://raw.githubusercontent.com/0xB10C/ofac-sanctioned-digital-currency-addresses/"
         "lists/sanctioned_addresses_{}.json")
def fetch_ofac(sym):
    with urllib.request.urlopen(_OFAC.format(sym), timeout=30) as r:
        return json.load(r)

# EVM seeds (ETH/USDT) — lowercase to match crypto_ethereum
SANCTIONED_ETH = {a.lower() for a in fetch_ofac("ETH")} | {a.lower() for a in fetch_ofac("USDT")}

# Tron base58 -> hex (EVM-style 20 bytes; Tron addresses are 0x41 + 20 + 4-byte checksum)
_B58 = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
def b58_to_hex(s):
    n = 0
    for ch in s: n = n*58 + _B58.index(ch)
    return "0x" + n.to_bytes(25, "big")[1:21].hex()
SANCTIONED_TRON = set()
for a in fetch_ofac("TRX"):
    try: SANCTIONED_TRON.add(b58_to_hex(a))
    except Exception: pass
print(f"OFAC seeds — ETH/USDT: {len(SANCTIONED_ETH)} | Tron: {len(SANCTIONED_TRON)}")

## 4. Candidate detection — Ethereum USDT (ERC-20), with dust floor

High fan-in / low fan-out, meaningful volume, **median per-transfer ≥ dust floor**, excluding known venues.

In [ ]:
known_eth = ", ".join(f"'{a}'" for a in KNOWN_EXCHANGE_ADDRESSES_ETH)
eth_sql = f"""
DECLARE usdt STRING DEFAULT '{USDT_ERC20}';
DECLARE since TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {LOOKBACK_DAYS} DAY);
WITH transfers AS (
  SELECT from_address, to_address,
         SAFE_CAST(value AS BIGNUMERIC)/POW(10,{USDT_DECIMALS}) amt, block_timestamp
  FROM `{ETH_TOKEN_TRANSFERS}`
  WHERE token_address=usdt AND block_timestamp>=since
),
incoming AS (
  SELECT to_address wallet, COUNT(*) in_cnt, COUNT(DISTINCT from_address) distinct_senders,
         SUM(amt) in_usdt, APPROX_QUANTILES(amt,2)[OFFSET(1)] in_median_usdt,
         MIN(block_timestamp) first_in, MAX(block_timestamp) last_in
  FROM transfers GROUP BY wallet
),
outgoing AS (
  SELECT from_address wallet, COUNT(*) out_cnt, COUNT(DISTINCT to_address) distinct_recipients,
         SUM(amt) out_usdt FROM transfers GROUP BY wallet
)
SELECT i.wallet, i.in_cnt, i.distinct_senders, i.in_usdt, i.in_median_usdt,
       COALESCE(o.out_cnt,0) out_cnt, COALESCE(o.distinct_recipients,0) distinct_recipients,
       COALESCE(o.out_usdt,0) out_usdt,
       SAFE_DIVIDE(i.in_cnt,COALESCE(o.out_cnt,0)) in_out_tx_ratio,
       TIMESTAMP_DIFF(i.last_in,i.first_in,HOUR) active_hours
FROM incoming i LEFT JOIN outgoing o USING(wallet)
WHERE i.distinct_senders>={MIN_DISTINCT_SENDERS}
  AND COALESCE(o.out_cnt,0)<={MAX_OUT_TX}
  AND i.in_usdt>={MIN_IN_USDT}
  AND i.in_median_usdt>={MIN_MEDIAN_USDT}     -- dust floor
  AND i.wallet NOT IN ({known_eth})
ORDER BY i.distinct_senders DESC, i.in_usdt DESC
LIMIT {RESULT_LIMIT}
"""
estimate_cost(eth_sql)   # dry-run first; then run the next cell

In [ ]:
eth = run_query(eth_sql)
eth["chain"] = "ethereum"
print("Ethereum USDT candidates (dust-floored):", len(eth))
eth.head(10)

### Drop smart contracts → keep EOA (people-operated) wallets only

In [ ]:
def drop_contracts_eth(df):
    addrs = df["wallet"].tolist()
    if not addrs: return df
    sql = f"SELECT address FROM `{ETH_CONTRACTS}` WHERE address IN ({', '.join(repr(a) for a in addrs)})"
    contracts = set(run_query(sql)["address"])
    out = df[~df["wallet"].isin(contracts)].copy()
    print(f"Dropped {len(contracts)} contracts -> {len(out)} EOAs")
    return out
eth_eoa = drop_contracts_eth(eth)

## 5. Candidate detection — Tron USDT (TRC-20)

Tron transfers live in `decoded_events`: `args` JSON = `[from, to, value]`, `address` = token contract (hex),
`event_signature = 'Transfer(address,address,uint256)'`. Same funnel logic + dust floor. Tron carries the bulk
of real USDT volume, so set a shorter window if cost matters (decoded_events ≈ 0.5 GB/day for USDT).

In [ ]:
TRON_DAYS = int(os.environ.get("TRON_LOOKBACK_DAYS", 7))   # keep modest; Tron USDT volume is huge
tron_sql = f"""
DECLARE since TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {TRON_DAYS} DAY);
WITH transfers AS (
  SELECT JSON_VALUE(args,'$[0]') from_address, JSON_VALUE(args,'$[1]') to_address,
         SAFE_CAST(JSON_VALUE(args,'$[2]') AS BIGNUMERIC)/POW(10,{USDT_DECIMALS}) amt, block_timestamp
  FROM `{TRON_EVENTS}`
  WHERE address='{USDT_TRC20_HEX}' AND event_signature='Transfer(address,address,uint256)'
    AND block_timestamp>=since
),
incoming AS (
  SELECT to_address wallet, COUNT(*) in_cnt, COUNT(DISTINCT from_address) distinct_senders,
         SUM(amt) in_usdt, APPROX_QUANTILES(amt,2)[OFFSET(1)] in_median_usdt,
         MIN(block_timestamp) first_in, MAX(block_timestamp) last_in
  FROM transfers GROUP BY wallet
),
outgoing AS (
  SELECT from_address wallet, COUNT(*) out_cnt, COUNT(DISTINCT to_address) distinct_recipients,
         SUM(amt) out_usdt FROM transfers GROUP BY wallet
)
SELECT i.wallet, i.in_cnt, i.distinct_senders, i.in_usdt, i.in_median_usdt,
       COALESCE(o.out_cnt,0) out_cnt, COALESCE(o.distinct_recipients,0) distinct_recipients,
       COALESCE(o.out_usdt,0) out_usdt,
       SAFE_DIVIDE(i.in_cnt,COALESCE(o.out_cnt,0)) in_out_tx_ratio,
       TIMESTAMP_DIFF(i.last_in,i.first_in,HOUR) active_hours
FROM incoming i LEFT JOIN outgoing o USING(wallet)
WHERE i.distinct_senders>={MIN_DISTINCT_SENDERS}
  AND COALESCE(o.out_cnt,0)<={MAX_OUT_TX}
  AND i.in_usdt>={MIN_IN_USDT}
  AND i.in_median_usdt>={MIN_MEDIAN_USDT}
ORDER BY i.distinct_senders DESC, i.in_usdt DESC
LIMIT {RESULT_LIMIT}
"""
estimate_cost(tron_sql)

In [ ]:
tron = run_query(tron_sql)
tron["chain"] = "tron"
# Tron addresses in decoded_events have no contract-flag table here; EOA filtering is left as a follow-up.
print("Tron USDT candidates (dust-floored):", len(tron))
tron.head(10)

## 6. Build the transaction graph (Layer 2)

Pull the transfers that touch the candidate wallets and build a directed weighted graph (nodes = wallets,
edges = aggregated flows). Shown for Ethereum; the same pattern applies to Tron.

In [ ]:
import networkx as nx

def fetch_edges_eth(wallets):
    wl = ", ".join(repr(w) for w in wallets)
    sql = f"""
    DECLARE usdt STRING DEFAULT '{USDT_ERC20}';
    DECLARE since TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {LOOKBACK_DAYS} DAY);
    SELECT from_address, to_address,
           SAFE_CAST(value AS BIGNUMERIC)/POW(10,{USDT_DECIMALS}) amt, block_timestamp
    FROM `{ETH_TOKEN_TRANSFERS}`
    WHERE token_address=usdt AND block_timestamp>=since
      AND (from_address IN ({wl}) OR to_address IN ({wl}))
    """
    estimate_cost(sql)
    return run_query(sql)

def build_graph(edges):
    agg = (edges.groupby(["from_address","to_address"])
           .agg(total_usdt=("amt","sum"), tx_count=("amt","size")).reset_index())
    g = nx.DiGraph()
    for r in agg.itertuples(index=False):
        g.add_edge(r.from_address, r.to_address, total_usdt=float(r.total_usdt), tx_count=int(r.tx_count))
    return g

# edges = fetch_edges_eth(eth_eoa["wallet"].tolist())
# G = build_graph(edges); print(G)

## 7. Features (Layer 3) — graph + mixer/funnel + sanctioned proximity

In [ ]:
def hops_to_sanctioned(g, seeds, max_hops=3):
    seeds = [s for s in seeds if s in g]
    if not seeds: return {}
    ug = g.to_undirected(as_view=True); dist = {s:0 for s in seeds}; cur = list(seeds); d = 0
    while cur and d < max_hops:
        d += 1; nxt = []
        for n in cur:
            for nb in ug.neighbors(n):
                if nb not in dist: dist[nb] = d; nxt.append(nb)
        cur = nxt
    return dist

def graph_features(g, seeds):
    pr = nx.pagerank(g, weight="total_usdt") if g.number_of_edges() else {}
    clus = nx.clustering(g.to_undirected()) if g.number_of_nodes() else {}
    rows = []
    for n in g.nodes:
        ie = list(g.in_edges(n, data=True)); oe = list(g.out_edges(n, data=True))
        rows.append(dict(wallet=n, pagerank=pr.get(n,0.0),
            in_degree=g.in_degree(n), out_degree=g.out_degree(n), clustering=clus.get(n,0.0),
            in_usdt=sum(d["total_usdt"] for *_,d in ie), out_usdt=sum(d["total_usdt"] for *_,d in oe),
            in_tx=sum(d["tx_count"] for *_,d in ie), out_tx=sum(d["tx_count"] for *_,d in oe)))
    df = pd.DataFrame(rows)
    dbal = 1-(abs(df.in_degree-df.out_degree)/(df.in_degree+df.out_degree).replace(0,np.nan))
    vbal = 1-(abs(df.in_usdt-df.out_usdt)/(df.in_usdt+df.out_usdt).replace(0,np.nan))
    multi = ((df.in_degree>=3)&(df.out_degree>=3)).astype(float)
    df["mixer_score"] = (dbal.fillna(0)*vbal.fillna(0)*multi).clip(0,1)
    df["funnel_score"] = np.log1p(df.in_degree)*(1/(1+df.out_degree))
    hops = hops_to_sanctioned(g, seeds, 3)
    df["hops_to_sanctioned"] = df.wallet.map(hops)
    df["is_sanctioned"] = df.wallet.isin(seeds)
    return df

# feats = graph_features(G, SANCTIONED_ETH)

## 8. Unsupervised anomaly scoring (Layer 3) → risk score

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = ["pagerank","in_degree","out_degree","clustering","in_usdt","out_usdt",
                "in_tx","out_tx","mixer_score","funnel_score"]
def score(df, contamination=0.05):
    X = df[FEATURE_COLS].copy()
    for c in ["in_usdt","out_usdt","in_tx","out_tx","pagerank"]: X[c] = np.log1p(X[c].astype(float))
    X = StandardScaler().fit_transform(X.replace([np.inf,-np.inf],np.nan).fillna(0))
    iso = IsolationForest(n_estimators=300, contamination=contamination, random_state=42).fit(X)
    out = df.copy(); out["anomaly_score"] = iso.decision_function(X); out["is_anomaly"] = iso.predict(X)==-1
    anom = (-out["anomaly_score"]).rank(pct=True)
    prox = (1/(1+out["hops_to_sanctioned"].fillna(99))).clip(0,1)
    out["risk_score"] = (60*anom + 40*prox).round(1)
    out.loc[out["is_sanctioned"]==True, "risk_score"] = 100.0
    return out.sort_values("risk_score", ascending=False)

# scored = score(feats)

## 9. Enrichment (Layer 1+) — human-readable labels (no API key)

From the community `brianleect/etherscan-labels` dataset (≈30k addresses). Verified to resolve e.g.
`0x28c6…1d60` → "Binance 14". In the executed run, 0/971 leads matched — the top funnels are *unlabeled*.

In [ ]:
_LAB = ("https://raw.githubusercontent.com/brianleect/etherscan-labels/"
        "main/data/etherscan/combined/combinedAllLabels.json")
def load_labels():
    with urllib.request.urlopen(_LAB, timeout=60) as r:
        return {k.lower(): v for k, v in json.load(r).items()}
def enrich(df):
    labels = load_labels(); a = df["wallet"].str.lower()
    out = df.copy()
    out["label_name"] = a.map(lambda x: (labels.get(x) or {}).get("name",""))
    out["label_tags"] = a.map(lambda x: ", ".join((labels.get(x) or {}).get("labels",[])))
    return out
# scored = enrich(scored)

## 10. CTF diagnostic — are OFAC-listed addresses even active?

The decisive counter-terror-financing check: scan *all* USDT transfers in the window and count how many touch
any OFAC seed. Executed results: **Ethereum** 1 transfer / 90d; **Tron** 1 of 71M / 30d. Listed addresses are
frozen/dormant — so direct proximity is a dead end; pivot to behavioural/upstream tracing.

In [ ]:
def ofac_activity_eth(days=90):
    seeds = ", ".join(repr(s) for s in SANCTIONED_ETH)
    sql = f"""
    DECLARE usdt STRING DEFAULT '{USDT_ERC20}';
    DECLARE since TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {days} DAY);
    SELECT COUNT(*) seed_touching_transfers,
           COUNT(DISTINCT IF(from_address IN ({seeds}), from_address, to_address)) active_seed_addrs
    FROM `{ETH_TOKEN_TRANSFERS}`
    WHERE token_address=usdt AND block_timestamp>=since
      AND (from_address IN ({seeds}) OR to_address IN ({seeds}))
    """
    estimate_cost(sql); return run_query(sql)

def ofac_activity_tron(days=30):
    seeds = ", ".join(repr(s) for s in SANCTIONED_TRON)
    sql = f"""
    DECLARE since TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {days} DAY);
    WITH t AS (
      SELECT JSON_VALUE(args,'$[0]') f, JSON_VALUE(args,'$[1]') tt
      FROM `{TRON_EVENTS}`
      WHERE address='{USDT_TRC20_HEX}' AND event_signature='Transfer(address,address,uint256)'
        AND block_timestamp>=since)
    SELECT COUNT(*) total_transfers,
           COUNTIF(f IN ({seeds}) OR tt IN ({seeds})) sanctioned_touching
    FROM t
    """
    estimate_cost(sql); return run_query(sql)

# ofac_activity_eth(90); ofac_activity_tron(30)

## 11. Visualise + export

In [ ]:
import matplotlib.pyplot as plt
def plot_candidates(df):
    fig, ax = plt.subplots(1, 2, figsize=(13,4.5))
    ax[0].scatter(df["distinct_senders"], df["out_cnt"], s=10, alpha=0.5)
    ax[0].set(xscale="log", xlabel="distinct senders (log)", ylabel="outgoing tx",
              title="Fan-in vs fan-out (funnel = bottom-right)")
    if "risk_score" in df: ax[1].hist(df["risk_score"], bins=30); ax[1].set(title="Risk score")
    plt.tight_layout(); plt.show()

# all_candidates = pd.concat([eth_eoa, tron], ignore_index=True)
# all_candidates.to_csv("usdt_funnel_candidates.csv", index=False)
# plot_candidates(all_candidates)

## 13. Identity deepening / attribution (Layers 1–2)

The wallet is just a hex string — these cells turn it into *entities and anchors* (never a magic "name"
field). All on-chain & legal. Four tools:
1. **Bitcoin co-spend clustering** — merge addresses owned by the same entity (common-input heuristic).
2. **Nearest-exchange anchor** — which regulated venue a candidate deposits to (the subpoena / KYC point).
3. **Behavioural fingerprint** — activity-hour histogram (→ timezone) + round-amount ratio.
4. **ENS resolution** — map ETH addresses to `.eth` names (off-chain pivot to a handle).

> ⚖️ **Boundary:** this produces *entities, anchors, and risk* — methodology, not accusations. Attaching a
> real person's name is for authorised enforcement (subpoena to the exchange that holds the KYC). Don't dox.

### 13.1 Bitcoin co-spend clustering (common-input-ownership heuristic)

All input addresses spent **together in one transaction** are controlled by the same entity. We map each
address to a representative (min co-spent address) — a fast 1-pass approximation of union-find. True transitive
clustering needs iteration, but this already collapses many addresses into entities.

> Cost: `crypto_bitcoin.inputs` is large; keep the `block_timestamp_month` partition filter. Dry-run first.

In [ ]:
btc_cluster_sql = f"""
DECLARE since      TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {LOOKBACK_DAYS} DAY);
DECLARE since_mon  DATE      DEFAULT DATE_SUB(CURRENT_DATE(), INTERVAL {LOOKBACK_DAYS} DAY);
WITH multi AS (
  SELECT t.`hash` AS transaction_hash, addr
  FROM `bigquery-public-data.crypto_bitcoin.transactions` t,
       UNNEST(t.inputs) inp,
       UNNEST(inp.addresses) addr
  WHERE t.block_timestamp_month >= since_mon
    AND t.block_timestamp >= since
    AND ARRAY_LENGTH(t.inputs) >= 2
),
tx_rep AS (
  SELECT transaction_hash, MIN(addr) AS entity_rep
  FROM multi GROUP BY transaction_hash HAVING COUNT(DISTINCT addr) >= 2
),
addr_entity AS (
  SELECT m.addr AS address, MIN(r.entity_rep) AS entity
  FROM multi m JOIN tx_rep r USING (transaction_hash)
  GROUP BY m.addr
)
SELECT entity,
       COUNT(*)                  AS n_addresses,
       ARRAY_AGG(address LIMIT 5) AS sample_addresses
FROM addr_entity
GROUP BY entity
ORDER BY n_addresses DESC
LIMIT 1000
"""
estimate_cost(btc_cluster_sql)
# btc_entities = run_query(btc_cluster_sql); btc_entities.head(20)

### 13.2 Nearest-exchange anchor (ETH)

For each candidate, find **direct** transfers to/from a labelled exchange. "Sends to Binance/OKX" = the place
that holds the customer's KYC → the real-world subpoena anchor. The exchange set is built from the community
Etherscan labels (Section 9).

In [ ]:
# Build a broad exchange-address set from the labels dataset (tags that imply a regulated venue).
EXCHANGE_TAGS = {"exchange","binance","coinbase","kraken","okx","kucoin","bitfinex","huobi","bybit",
                 "gate.io","crypto-com","bitstamp","gemini","mexc","bitget"}
def exchange_addresses():
    labels = load_labels()
    return {a for a, v in labels.items() if set(v.get("labels", [])) & EXCHANGE_TAGS}

def nearest_exchange_sql(candidate_addrs, exchange_addrs):
    cand = ", ".join(repr(a) for a in candidate_addrs)
    exch = ", ".join(repr(a) for a in exchange_addrs)
    return f"""
DECLARE usdt STRING DEFAULT '{USDT_ERC20}';
DECLARE since TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {LOOKBACK_DAYS} DAY);
SELECT
  IF(from_address IN ({cand}), from_address, to_address)              AS candidate,
  IF(from_address IN ({exch}), 'received_from_exchange', 'sent_to_exchange') AS direction,
  IF(from_address IN ({exch}), from_address, to_address)              AS exchange_addr,
  COUNT(*)                                                            AS n_tx,
  SUM(SAFE_CAST(value AS BIGNUMERIC)/POW(10,{USDT_DECIMALS}))         AS usdt
FROM `{ETH_TOKEN_TRANSFERS}`
WHERE token_address = usdt AND block_timestamp >= since
  AND ( (from_address IN ({cand}) AND to_address   IN ({exch}))
     OR (from_address IN ({exch}) AND to_address   IN ({cand})) )
GROUP BY candidate, direction, exchange_addr
ORDER BY usdt DESC
"""
# ex = exchange_addresses(); print("exchange addrs:", len(ex))
# anchor_sql = nearest_exchange_sql(eth_eoa["wallet"].tolist(), ex)
# estimate_cost(anchor_sql)
# anchors = run_query(anchor_sql); anchors.head(20)

### 13.3 Behavioural fingerprint — active hours (→ timezone) + round amounts

Activity by UTC hour reveals the operator's likely timezone (people sleep). A high share of round amounts
(e.g. exact multiples of 100/1000 USDT) suggests manual/OTC operation rather than automated flow.

In [ ]:
def fingerprint_sql(candidate_addrs):
    cand = ", ".join(repr(a) for a in candidate_addrs)
    return f"""
DECLARE usdt STRING DEFAULT '{USDT_ERC20}';
DECLARE since TIMESTAMP DEFAULT TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {LOOKBACK_DAYS} DAY);
SELECT to_address AS wallet,
       EXTRACT(HOUR FROM block_timestamp) AS hour_utc,
       COUNT(*) AS n_tx,
       COUNTIF(MOD(SAFE_CAST(value AS BIGNUMERIC), CAST(100*POW(10,{USDT_DECIMALS}) AS BIGNUMERIC))=0) AS round_100s
FROM `{ETH_TOKEN_TRANSFERS}`
WHERE token_address = usdt AND block_timestamp >= since AND to_address IN ({cand})
GROUP BY wallet, hour_utc
"""
# fp = run_query(fingerprint_sql(eth_eoa["wallet"].tolist()))
# peak = (fp.sort_values("n_tx", ascending=False).groupby("wallet").first())
# peak["tz_guess_utc_offset"] = ((12 - peak.index.map(lambda w: int(peak.loc[w,"hour_utc"]))) % 24)  # crude midday-anchor
# peak.head(20)

### 13.4 ENS resolution (ETH address → `.eth` name)

Most reliable via the **ENS subgraph** (off-BigQuery). A primary `.eth` name often links to a Twitter/website
→ an OSINT pivot. Coverage is partial (not everyone sets a name).

In [ ]:
def ens_names(addresses):
    """Resolve ETH addresses -> .eth names via the ENS subgraph (best-effort, partial coverage)."""
    import urllib.request, json as _json
    url = "https://api.thegraph.com/subgraphs/name/ensdomains/ens"
    out = {}
    for a in addresses:
        q = {"query": '{domains(where:{resolvedAddress:"%s"} first:5){name}}' % a.lower()}
        try:
            req = urllib.request.Request(url, data=_json.dumps(q).encode(),
                                         headers={"Content-Type": "application/json"})
            data = _json.load(urllib.request.urlopen(req, timeout=20))
            names = [d["name"] for d in data.get("data", {}).get("domains", [])]
            if names: out[a] = names
        except Exception:
            pass
    return out
# ens = ens_names(eth_eoa["wallet"].head(50).tolist()); ens
# (Bulk alternative in BigQuery: decode NameRegistered events from crypto_ethereum.logs of the ENS
#  registrar controller — more work; the subgraph above is the practical path.)

### 13.5 Attributability score

Combine the anchors into one 0–100 "how identifiable" score per candidate, to prioritise which leads are
worth a deeper (lawful) investigation.

In [ ]:
def attributability(df, anchors=None, ens=None, labels=None):
    """+40 direct exchange anchor, +30 own label, +20 ENS name, +10 sanctioned-proximity."""
    s = pd.Series(0, index=df.index, dtype=float)
    w = df["wallet"].str.lower()
    if anchors is not None and len(anchors):
        anchored = set(anchors["candidate"].str.lower()); s += w.isin(anchored)*40
    if labels is not None:
        s += w.map(lambda x: 30 if (labels.get(x) or {}).get("name") else 0)
    if ens is not None:
        ens_l = {k.lower() for k in ens}; s += w.isin(ens_l)*20
    if "hops_to_sanctioned" in df:
        s += (df["hops_to_sanctioned"].fillna(99) <= 2)*10
    out = df.copy(); out["attributability"] = s.clip(0,100); return out.sort_values("attributability", ascending=False)
# scored = attributability(scored, anchors=anchors, ens=ens, labels=load_labels())

## 12. Limitations & responsible use

- **A pattern is not a crime.** High fan-in / low fan-out also fits payment processors, OTC desks, exchange
  deposit wallets, donation pools. Every match is a **research lead**, never proof.
- **Dust caveat handled, not eliminated.** The median dust floor removes most airdrop/spam funnels; still sanity-check value distributions.
- **CTF reality.** Already-listed OFAC addresses are frozen/dormant in USDT on both chains — direct proximity
  yields almost nothing. Real signal needs behavioural detection and upstream tracing, and authoritative seeds only.
- **Public data only.** No deanonymisation of real-world identities; operate within a lawful AML programme.
- **Cost discipline.** Always `estimate_cost()` before `run_query()`; keep the `block_timestamp` partition filter.